<a href="https://githubtocolab.com/tannialhernandez/topicosAvanzadosAnalitica/blob/main/E4-TransferLearning/E4_PretrainedModelsPytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tranfer Learnning to Flower Recognition using PyTorch 🔥
This dataset contains labeled 4242 images of flowers.

### Content
The pictures are divided into five classes: chamomile, tulip, rose, sunflower, dandelion.

For each class there are about 800 photos. Photos are not high resolution, about 320x240 pixels. Photos are not reduced to a single size, they have different proportions!

### Data
You can download data from: [Flowers Recognition Dataset](https://www.kaggle.com/datasets/alxmamaev/flowers-recognition)

In [71]:
import subprocess
import sys

required_packages = [
    'torch',
    'torchvision',
    'torchmetrics',
    'numpy',
    'gdown'
]

def install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

# Check if the libraries are installed and install them if they are not.
for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        install(package)

In [75]:
import zipfile
import random
import shutil
import torch
import os
import numpy as np
import torch.nn as nn
import torch.optim as optim
import gdown
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchmetrics import AUROC

In [76]:
file_id = '1C65mg3uzQ4UGl0mjsUa6QsNtvfkPD4nj'
url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(url, 'archive.zip', quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1C65mg3uzQ4UGl0mjsUa6QsNtvfkPD4nj
From (redirected): https://drive.google.com/uc?id=1C65mg3uzQ4UGl0mjsUa6QsNtvfkPD4nj&confirm=t&uuid=a337392b-24d4-47eb-bfbf-dcb749b940dc
To: /content/archive.zip
100%|██████████| 236M/236M [00:02<00:00, 89.7MB/s]


'archive.zip'

In [77]:
path="./"
os.listdir(path)
zip_file_path = "archive.zip"
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(path)

In [5]:
path="./flowers"
os.listdir(path)

['sunflower', 'tulip', 'daisy', 'dandelion', 'rose']

In [6]:
#Definition of data transformations for data augmentation and normalization
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [7]:
# Define the path to the folder 'flowers' and create the folder 'dataset'.
dataset_dir = './dataset'  # Folder in which 'train' and 'val' are to be created
train_dir = os.path.join(dataset_dir, 'train')
val_dir = os.path.join(dataset_dir, 'val')
split_ratio = 0.8  # Training ratio

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Valid image file extensions
valid_extensions = ['.jpg', '.jpeg', '.png', '.gif']

for class_name in os.listdir(path):
    class_path = os.path.join(path, class_name)

    if os.path.isdir(class_path):
        os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
        os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)

        images = [img for img in os.listdir(class_path) if os.path.splitext(img)[1].lower() in valid_extensions]
        random.shuffle(images)

        # Calculating the number of images for training and validation
        split_index = int(len(images) * split_ratio)
        train_images = images[:split_index]
        val_images = images[split_index:]

        for img in train_images:
            shutil.copy(os.path.join(class_path, img), os.path.join(train_dir, class_name, img))

        for img in val_images:
            shutil.copy(os.path.join(class_path, img), os.path.join(val_dir, class_name, img))

print("Data divided into 'train' and 'val' folders within 'dataset'.")

Data divided into 'train' and 'val' folders within 'dataset'.


In [8]:
# Create data loaders
image_datasets = {x: datasets.ImageFolder(os.path.join(dataset_dir, x), data_transforms[x]) for x in ['train', 'val']}

In [58]:
# Using dataloader
dataloaders = {x: DataLoader(image_datasets[x], batch_size= 4, shuffle=True, num_workers=4) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
print(dataset_sizes)
class_names = image_datasets['train'].classes
class_names

{'train': 4149, 'val': 1562}


['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']

In [59]:
#Load the pre-trained ResNet-18 model
model = models.resnet18(pretrained=True)

#Freeze all layers except the final classsification layer
for name, param in model.named_parameters():
  #Fully Connected Layer
  if "fc" in name: #Unfreeze the final classification layer
    param.requires_grad = True
  else:
    param.requires_grad = False

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [60]:
#Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9) #Use all parameters

# Move the model to the GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [65]:
# train model and evaluate model
def train_eval_model(model, criterion, optimizer, num_epochs=25):

  auc_metric = AUROC(num_classes=len(class_names), task='multiclass').to(device)

  for epoch in range(num_epochs):
    for phase in ['train', 'val']:
      if phase == 'train':
        model.train()
      else:
        model.eval()

      running_loss = 0.0
      running_corrects = 0
      all_labels = []
      all_probs = []

      for inputs, labels in dataloaders[phase]:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        with torch.set_grad_enabled(phase == 'train'):
          outputs = model(inputs)
          probs = torch.nn.functional.softmax(outputs[:, :len(class_names)], dim=1)
          _, preds = torch.max(outputs, 1)
          loss = criterion(outputs, labels)

          if phase == 'val':
            all_labels.append(labels.cpu())
            all_probs.append(probs.cpu())

          if phase == 'train':
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

      epoch_loss = running_loss / dataset_sizes[phase]
      epoch_acc = running_corrects.double() / dataset_sizes[phase]

      print(f'Epoch {epoch+1}/{num_epochs} {phase} Loss: {epoch_loss:.4f} Accuracy: {epoch_acc:.4f}')

      if phase == 'val':
        # Concatenar todas las etiquetas y probabilidades
        all_labels = torch.cat(all_labels)
        all_probs = torch.cat(all_probs)

        # Calcular AUC
        auc_score = auc_metric(all_probs, all_labels)
        print(f'AUC: {auc_score:.4f}')

print("Training complete!")

Training complete!


In [62]:
# training
train_eval_model(model, criterion, optimizer, num_epochs=25)

Epoch 1/25 train Loss: 1.1986 Accuracy: 0.5958
Epoch 1/25 val Loss: 0.4815 Accuracy: 0.8323
AUC: 0.9760
Epoch 2/25 train Loss: 0.8724 Accuracy: 0.6932
Epoch 2/25 val Loss: 0.4355 Accuracy: 0.8521
AUC: 0.9798
Epoch 3/25 train Loss: 0.8482 Accuracy: 0.6975
Epoch 3/25 val Loss: 0.3831 Accuracy: 0.8771
AUC: 0.9833
Epoch 4/25 train Loss: 0.8518 Accuracy: 0.7043
Epoch 4/25 val Loss: 0.3521 Accuracy: 0.8803
AUC: 0.9846
Epoch 5/25 train Loss: 0.8416 Accuracy: 0.7127
Epoch 5/25 val Loss: 0.4056 Accuracy: 0.8675
AUC: 0.9825
Epoch 6/25 train Loss: 0.8097 Accuracy: 0.7192
Epoch 6/25 val Loss: 0.3408 Accuracy: 0.8905
AUC: 0.9854
Epoch 7/25 train Loss: 0.8190 Accuracy: 0.7209
Epoch 7/25 val Loss: 0.3510 Accuracy: 0.8892
AUC: 0.9844
Epoch 8/25 train Loss: 0.7773 Accuracy: 0.7279
Epoch 8/25 val Loss: 0.3739 Accuracy: 0.8822
AUC: 0.9860
Epoch 9/25 train Loss: 0.7903 Accuracy: 0.7293
Epoch 9/25 val Loss: 0.3511 Accuracy: 0.8848
AUC: 0.9851
Epoch 10/25 train Loss: 0.8522 Accuracy: 0.7149
Epoch 10/25 val 

## Transfer Learning
By using a pre-trained model, it is not necessary to have large amounts of labeled data.

* We retrain a random number of layers of the pretrained model to adjust the weights to our specific data set.
* We freeze the initial layers (which generally learn more general features) and only allow the later layers to be updated during training. This helps to avoid overfitting.

### Note

The weights='IMAGENET1K_V1' argument instead of pretrained=True is part of an update in the torchvision library that was implemented starting with its 0.13 version. This update provides a clearer and more explicit way to handle pretrained weights, and also allows to choose between different pretrained weights that may be suitable for various tasks and datasets.



In [69]:
model = models.resnet18(weights='IMAGENET1K_V1')
frozen_count = 0

num_layers_to_freeze = random.randint(5, 20)  # Freezing between 5 and 20 layers

for name, param in model.named_parameters():
    if frozen_count < num_layers_to_freeze:
        param.requires_grad = False
        frozen_count += 1
    else:
        break

# Unfreeze the final classification layer
for name, param in model.named_parameters():
    if "fc" in name:
        param.requires_grad = True

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, momentum=0.9)

# Move the model to the GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f'Number of frozen layers: {num_layers_to_freeze}')


Number of frozen layers: 16


In [70]:
# training
train_eval_model(model, criterion, optimizer, num_epochs=25)

Epoch 1/25 train Loss: 1.1071 Accuracy: 0.6602
Epoch 1/25 val Loss: 0.3995 Accuracy: 0.8707
AUC: 0.9866
Epoch 2/25 train Loss: 0.7216 Accuracy: 0.7590
Epoch 2/25 val Loss: 0.2492 Accuracy: 0.9200
AUC: 0.9926
Epoch 3/25 train Loss: 0.6530 Accuracy: 0.7727
Epoch 3/25 val Loss: 0.1999 Accuracy: 0.9238
AUC: 0.9958
Epoch 4/25 train Loss: 0.5791 Accuracy: 0.8065
Epoch 4/25 val Loss: 0.1834 Accuracy: 0.9392
AUC: 0.9962
Epoch 5/25 train Loss: 0.5755 Accuracy: 0.8045
Epoch 5/25 val Loss: 0.1969 Accuracy: 0.9385
AUC: 0.9947
Epoch 6/25 train Loss: 0.5001 Accuracy: 0.8383
Epoch 6/25 val Loss: 0.2243 Accuracy: 0.9264
AUC: 0.9942
Epoch 7/25 train Loss: 0.4749 Accuracy: 0.8426
Epoch 7/25 val Loss: 0.1879 Accuracy: 0.9392
AUC: 0.9952
Epoch 8/25 train Loss: 0.4653 Accuracy: 0.8465
Epoch 8/25 val Loss: 0.1755 Accuracy: 0.9398
AUC: 0.9968
Epoch 9/25 train Loss: 0.4134 Accuracy: 0.8624
Epoch 9/25 val Loss: 0.2082 Accuracy: 0.9283
AUC: 0.9954
Epoch 10/25 train Loss: 0.4360 Accuracy: 0.8672
Epoch 10/25 val 

## Conclusions

* By employing a pre-trained model such as ResNet-18, more efficient training is achieved, leading to an accuracy of up to 97.44% on the validation set and an AUC of up to 0.9987 in only 25 epochs. This highlights the model's ability to effectively discriminate between classes.
* By using features already learned from a large dataset (ImageNet), the model was able to capture patterns relevant to flower classification, thus outperforming models trained from scratch.

